# scrape judul dan url


In [4]:
import requests
import urllib3
import pandas as pd
from bs4 import BeautifulSoup

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

def scrape_kominfo():

    data = []

    for page_num in range(1, 4):

        url = (
            f"https://kominfo.malangkab.go.id/berita?page={page_num}"
        )

        print(f"scraping page {page_num}")

        html = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0"
            },
            verify=False,
            timeout=30
        ).text

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        for article in soup.select(
            "div.bg-white.rounded-xl.shadow-lg"
        ):

            title_tag = article.select_one("h3")

            link_tag = article.select_one(
                "a[href*='/berita/']"
            )

            date_tag = article.select_one(
                "p.text-sm.text-gray-500"
            )

            if not title_tag or not link_tag:
                continue

            data.append({
                "title": title_tag.get_text(strip=True),
                "url": link_tag["href"],
                "published_date": (
                    date_tag.get_text(strip=True)
                    if date_tag else None
                )
            })

    df = (
        pd.DataFrame(data)
        .drop_duplicates(subset=["url"])
        .reset_index(drop=True)
    )

    print(df.head())
    print(f"Found {len(df)} articles")

    df.to_csv(
        "kominfo_malangkab.csv",
        index=False
    )

    return df

df_kominfo = scrape_kominfo()

scraping page 1
scraping page 2
scraping page 3
                                               title  \
0  Mengenal Konsep AI Sandwich: Cara Bijak Menggu...   
1  KIM of the Week: KIM Karya Inovatif Angkat Pen...   
2  Diskominfo Kabupaten Malang Gelar Sosialisasi ...   
3  SIM Digital, Solusi Praktis Berkendara di Era ...   
4  KIM of the Week: KIM Sitirejo Angkat Inovasi K...   

                                                 url     published_date  
0  http://kominfo.malangkab.go.id/berita/mengenal...  17 June 26, 09:20  
1  http://kominfo.malangkab.go.id/berita/kim-of-t...  17 June 26, 09:05  
2  http://kominfo.malangkab.go.id/berita/diskomin...  15 June 26, 14:10  
3  http://kominfo.malangkab.go.id/berita/sim-digi...  10 June 26, 09:22  
4  http://kominfo.malangkab.go.id/berita/kim-of-t...  04 June 26, 15:38  
Found 27 articles


# scrape isi url 

In [5]:
import requests
import urllib3
import pandas as pd
from bs4 import BeautifulSoup

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

# hasil dari scrape list artikel
df = pd.read_csv("kominfo_malangkab.csv")

articles = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for i, row in df.iterrows():

    try:

        html = requests.get(
            row["url"],
            headers=headers,
            verify=False,
            timeout=30
        ).text

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        title = (
            soup.select_one("h1")
            .get_text(strip=True)
            if soup.select_one("h1")
            else None
        )

        date = (
            soup.select_one(
                "p.text-sm.text-gray-500"
            )
            .get_text(" ", strip=True)
            if soup.select_one(
                "p.text-sm.text-gray-500"
            )
            else None
        )

        image = (
            soup.select_one(
                "div.mb-6 img"
            )["src"]
            if soup.select_one(
                "div.mb-6 img"
            )
            else None
        )

        content_div = soup.select_one(
            "div.prose"
        )

        content = None

        if content_div:

            content = "\n".join([
                p.get_text(
                    " ",
                    strip=True
                )
                for p in content_div.select("p")
            ])

        articles.append({
            "title": title,
            "published_date": date,
            "image_url": image,
            "content": content,
            "url": row["url"],
            "source": "kominfo_malangkab"
        })

        print(
            f"[{i+1}/{len(df)}] success"
        )

    except Exception as e:

        print(
            f"[{i+1}/{len(df)}] failed: {e}"
        )

result = pd.DataFrame(articles)

result.to_csv(
    "kominfo_malangkab_articles.csv",
    index=False
)

print(
    f"saved {len(result)} articles"
)

result.head()

[1/27] success
[2/27] success
[3/27] success
[4/27] success
[5/27] success
[6/27] success
[7/27] success
[8/27] success
[9/27] success
[10/27] success
[11/27] success
[12/27] success
[13/27] success
[14/27] success
[15/27] success
[16/27] success
[17/27] success
[18/27] success
[19/27] success
[20/27] success
[21/27] success
[22/27] success
[23/27] success
[24/27] success
[25/27] success
[26/27] success
[27/27] success
saved 27 articles


,title,published_date,image_url,content,url,source
0,Mengenal Konsep AI Sandwich: Cara Bijak Menggu...,Navigasi,https://web-admin.malangkab.go.id/3//uploads/a...,Perkembangan kecerdasan buatan atau Artificial...,http://kominfo.malangkab.go.id/berita/mengenal...,kominfo_malangkab
1,KIM of the Week: KIM Karya Inovatif Angkat Pen...,Navigasi,https://web-admin.malangkab.go.id/3//uploads/a...,Komitmen dalam menyampaikan informasi yang ber...,http://kominfo.malangkab.go.id/berita/kim-of-t...,kominfo_malangkab
2,Diskominfo Kabupaten Malang Gelar Sosialisasi ...,Navigasi,https://web-admin.malangkab.go.id/3//uploads/a...,Dalam rangka mendukung implementasi Pemerintah...,http://kominfo.malangkab.go.id/berita/diskomin...,kominfo_malangkab
3,"SIM Digital, Solusi Praktis Berkendara di Era ...",Navigasi,https://web-admin.malangkab.go.id/3//uploads/a...,"Dengan SIM Digital, masyarakat tidak perlu pan...",http://kominfo.malangkab.go.id/berita/sim-digi...,kominfo_malangkab
4,KIM of the Week: KIM Sitirejo Angkat Inovasi K...,Navigasi,https://web-admin.malangkab.go.id/3//uploads/a...,Komitmen dalam menyampaikan informasi yang ins...,http://kominfo.malangkab.go.id/berita/kim-of-t...,kominfo_malangkab
